In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# -----------------------
# 1️⃣ Example Vocabulary
# -----------------------
verbs = {"your": 0, "is": 1, "cash": 2, "account": 3, "otp": 4, "credited": 5, "won": 6, "prize": 7}

# -----------------------
# 2️⃣ Example Dataset
# -----------------------

# Each message is converted to a vector (length = len(verbs))
data = [
    # OTP messages
    ([0.9, 0.8, 0.0, 0.1, 0.9, 0.0, 0.0, 0.0], 0),
    ([0.8, 0.9, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0], 0),

    # Transaction messages
    ([0.1, 0.0, 0.8, 0.9, 0.0, 0.9, 0.0, 0.0], 1),
    ([0.2, 0.0, 0.7, 0.8, 0.0, 1.0, 0.0, 0.0], 1),

    # Spam messages
    ([0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.9, 0.9], 2),
    ([0.1, 0.1, 0.0, 0.0, 0.0, 0.0, 0.8, 0.7], 2)
]

X = torch.tensor([x for x, _ in data], dtype=torch.float32)
y = torch.tensor([y for _, y in data], dtype=torch.long)

# -----------------------
# 3️⃣ Model
# -----------------------
class MessageClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.fc(x)

model = MessageClassifier(input_dim=len(verbs), num_classes=3)

# -----------------------
# 4️⃣ Train
# -----------------------
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

for epoch in range(300):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()

print(f"Training complete. Final loss: {loss.item():.4f}")

# -----------------------
# 5️⃣ Test
# -----------------------
test_input = torch.tensor([[0.9, 0.8, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]])  # new message (like "Your OTP is 3456")
pred = model(test_input)
label_index = torch.argmax(pred, dim=1).item()
labels = ["OTP", "Transaction", "Spam"]
print("Predicted Category:", labels[label_index])

# -----------------------
# 6️⃣ Export for Mobile
# -----------------------
model.eval()
example_input = torch.randn(1, len(verbs))
traced_script_module = torch.jit.trace(model, example_input)
traced_script_module.save("msg_model.pt")

from torch.utils.mobile_optimizer import optimize_for_mobile
optimized_model = optimize_for_mobile(traced_script_module)
optimized_model._save_for_lite_interpreter("msg_model.ptl")


Training complete. Final loss: 0.0194
Predicted Category: OTP


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# -----------------------
# 1️⃣ Example Vocabulary
# -----------------------
verbs = {"your": 0, "is": 1, "cash": 2, "account": 3, "otp": 4, "credited": 5, "won": 6, "prize": 7}

# -----------------------
# 2️⃣ Example Dataset
# -----------------------

# Each message is converted to a vector (length = len(verbs))
data = [
    # OTP messages
    ([0.9, 0.8, 0.0, 0.1, 0.9, 0.0, 0.0, 0.0], 0),
    ([0.8, 0.9, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0], 0),

    # Transaction messages
    ([0.1, 0.0, 0.8, 0.9, 0.0, 0.9, 0.0, 0.0], 1),
    ([0.2, 0.0, 0.7, 0.8, 0.0, 1.0, 0.0, 0.0], 1),

    # Spam messages
    ([0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.9, 0.9], 2),
    ([0.1, 0.1, 0.0, 0.0, 0.0, 0.0, 0.8, 0.7], 2)
]

X = torch.tensor([x for x, _ in data], dtype=torch.float32)
y = torch.tensor([y for _, y in data], dtype=torch.long)

# -----------------------
# 3️⃣ Model
# -----------------------
class MessageClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.fc(x)

model = MessageClassifier(input_dim=len(verbs), num_classes=3)

# -----------------------
# 4️⃣ Train
# -----------------------
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

for epoch in range(300):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()

print(f"Training complete. Final loss: {loss.item():.4f}")

Training complete. Final loss: 0.0199


In [1]:
#Testing model
pred = model(test_input)
label_index = torch.argmax(pred, dim=1).item()
labels = ["OTP", "Transaction", "Spam"]
print("Predicted Category:", labels[label_index])

NameError: name 'model' is not defined

In [ ]:
#Creating TFIDF vector
import torch
from sklearn.feature_extraction.text import TfidfVectorizer

# -------------------------------------------------
# 1️⃣  Your existing vocabulary (verbs list)
# -------------------------------------------------
verbs = {
    "your": 0,
    "is": 1,
    "cash": 2,
    "account": 3,
    "otp": 4,
    "credited": 5,
    "won": 6,
    "prize": 7
}

# -------------------------------------------------
# 2️⃣  Create a single reusable TfidfVectorizer
# -------------------------------------------------
vectorizer = TfidfVectorizer(vocabulary=list(verbs.keys()))

# Fit once on your full training corpus
# (Here we use a few sample messages; in real use, fit on all your training data)
corpus = [
    "Your OTP is 1234",
    "Your account has been credited with cash",
    "Congratulations you won a prize",
]
vectorizer.fit(corpus)

# -------------------------------------------------
# 3️⃣  Function: convert message → tensor
# -------------------------------------------------
def message_to_tfidf_tensor(message: str, vectorizer, vocab: dict):
    """
    Converts any text message into a torch tensor TF-IDF vector
    matching the order of your 'verbs' vocabulary.
    """
    # Transform the message using the fitted TF-IDF vectorizer
    X = vectorizer.transform([message.lower()])

    # Convert to numpy array
    tfidf_vector = X.toarray()[0]

    # Wrap in tensor for PyTorch model
    return torch.tensor([tfidf_vector], dtype=torch.float32)

# -------------------------------------------------
# 4️⃣  Example: convert a new message
# -------------------------------------------------
msg = "You have credited 1000 rs"
test_input = message_to_tfidf_tensor(msg, vectorizer, verbs)
print("TF-IDF Vector:", test_input)
print("Vector length:", test_input.shape[1])  # should match len(verbs)


TF-IDF Vector: tensor([[0., 0., 0., 0., 0., 1., 0., 0.]])
Vector length: 8


In [ ]:
#Exporting Model
model.eval()
example_input = torch.randn(1, len(verbs))
traced_script_module = torch.jit.trace(model, example_input)
traced_script_module.save("msg_model.pt")

from torch.utils.mobile_optimizer import optimize_for_mobiles
optimized_model = optimize_for_mobile(traced_script_module)
optimized_model._save_for_lite_interpreter("msg_model.ptl")